## Execution mode

This public notebook has two explicit execution paths:

- `public` verifies the signed, identifier-free research outputs released under
  `artifacts/` and materializes the corresponding figures and tables without
  CRSP data;
- `full` executes the original empirical notebook and requires the licensed
  inputs documented in `DATA_ACCESS.md`.

The default public path never reads security-level returns, holdings,
identifiers or portfolio weights. No silent fallback is performed.


In [ ]:
# Change to "full" only when the licensed local inputs are available.
import os
from pathlib import Path

RUN_MODE = os.environ.get("MFDRO_RUN_MODE", "public").strip().lower()
if RUN_MODE not in {"public", "full"}:
    raise ValueError("RUN_MODE must be 'public' or 'full'")

_start = Path.cwd().resolve()
PUBLIC_ROOT = next(
    (p for p in (_start, *_start.parents) if (p / "artifacts" / "SHA256SUMS").is_file()),
    None,
)
if PUBLIC_ROOT is None:
    raise FileNotFoundError("run the notebook from within the mt-mfdro repository")

import sys
if str(PUBLIC_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLIC_ROOT))

print(f"Execution mode: {RUN_MODE.upper()}")
print(f"Repository root: {PUBLIC_ROOT}")


In [ ]:
if RUN_MODE == "public":
    from src.public_results import run_public_notebook

    PUBLIC_REPORT = run_public_notebook("signal", root=PUBLIC_ROOT)


In [ ]:
if RUN_MODE == "full":
    print("FULL MODE — executing the original licensed-data workflow")


# Multi-Frequency Wasserstein Dispersion Signal — Validation

**Endogenous calibration of the Wasserstein ambiguity radius.**

This notebook validates the multi-frequency dispersion signal $\widehat{\rho}_{\tau}$, the input that drives the dynamic ambiguity radius of the DRO portfolio. Validation proceeds in order of increasing confidence: (i) the signal reacts to externally dated stress episodes, (ii) it discriminates a real distributional misalignment from finite-sample noise, and (iii) its dynamics are robust to the numerical choices of its construction.

The signal is read from the *validated, point-in-time* series published under `data/signals/` by `tests/02_validate_and_publish_pit_signals.ipynb`. Weekly and monthly returns are **geometrically compounded** — $(1+r)$ products, never sums — on the exactly-100-name formation membership built by `notebooks/01_build_pit_big_small_caps.ipynb`; no return is imputed. Sections requiring perturbed or repeated windows (placebo, power, ROC, Monte-Carlo) recompute $\widehat{\rho}_{\tau}$ with the *same* compounded construction and cache their outputs under `cache/signal/`.

Reference specification: rolling window $W=36$ months, free-support barycenter with $M=50$ atoms, uniform frequency weights, $L=200$ slicing projections. Sample: 1995–2025 (monthly).

## Setup

In [ ]:
if RUN_MODE == 'full':
    # Environment, clean plotting style, paths, IO helpers
    import numpy as np
    import pandas as pd
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from pathlib import Path

    # --- Paths (robust to CWD: works from repo root or notebooks/tests sub-dirs) ---
    ROOT = Path.cwd()
    if ROOT.name in {'notebooks', 'tests'}:
        ROOT = ROOT.parent
    DATA = ROOT/'data'
    SIG   = DATA/'signals'                                    # validated compounded signals (05 -> tests/02)
    PANEL = DATA/'processed'/'nyse_big_caps_pit_daily.parquet'  # PIT panel built by notebook 04
    VIX   = DATA/'processed'/'vix_1990.parquet'

    OUT = ROOT/'outputs'/'results'/'signal'; OUT.mkdir(parents=True, exist_ok=True)
    IMG = OUT/'images'; IMG.mkdir(parents=True, exist_ok=True)
    TAB = OUT/'tables'; TAB.mkdir(parents=True, exist_ok=True)

    # Recompute caches (placebo / power / Monte-Carlo), named in English, one sub-folder per family.
    # Internal working files live under the project-wide cache root, one folder per domain
    # (alongside notebook 02's engine cache), never under outputs/ which holds deliverables only.
    CACHE      = ROOT/'cache'/'signal'
    CACHE_PLAC = CACHE/'placebo';     CACHE_PLAC.mkdir(parents=True, exist_ok=True)
    CACHE_POW  = CACHE/'power';       CACHE_POW.mkdir(parents=True, exist_ok=True)
    CACHE_MC   = CACHE/'monte_carlo'; CACHE_MC.mkdir(parents=True, exist_ok=True)
    AUDIT      = OUT/'audit'; AUDIT.mkdir(parents=True, exist_ok=True)  # machine-readable trail (symmetry with 04)

    # Reference signal file: UNIFORM frequency weights (lambda_k = 1/3), the main spec.
    #   V_uni       -> big caps, W=36, uniform weights (REFERENCE)
    #   V_small_uni -> NYSE P20-P50, W=36, uniform weights
    #   U_*         -> robustness variants (cardinality, weights, scaling, distance, geometry)
    SIG_REF = 'V_uni.parquet'

    # --- Academic plotting style (Journal-of-Finance flavour) ---
    _NAVY, _RUST, _GREEN, _GREY = '#1f3b5c', '#a23b2e', '#2e6e4e', "#3c3c3c"
    PALETTE = [_NAVY, _RUST, _GREEN, '#b5882b', '#5c4b8a']
    mpl.rcParams.update({
        'figure.dpi'        : 120,
        'savefig.dpi'       : 300,
        'font.family'       : 'serif',
        'font.serif'        : ['Times New Roman','DejaVu Serif'],
        'font.size'         : 11,
        'axes.titlesize'    : 12,
        'axes.labelsize'    : 11,
        'axes.edgecolor'    : '#333333',
        'axes.linewidth'    : 0.8,
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.color'        : '#dddddd',
        'grid.linewidth'    : 0.6,
        'legend.frameon'    : False,
        'legend.fontsize'   : 10,
        'xtick.direction'   : 'out',
        'ytick.direction'   : 'out',
        'figure.facecolor'  : 'white',
        'axes.facecolor'    : 'white',
    })

    # --- Externally dated stress episodes (peak months) ---
    CRISES = [('Dot-com','2000-03'), ('GFC','2008-09'), ('Euro debt','2011-08'),
              ('COVID-19','2020-03'), ('Rate shock','2022-06')]

    # --- Fixed figure numbering + save helper ---
    # The number is an explicit identity of the figure (R_01 is always R_01),
    # NOT a running counter: re-running a cell never renumbers anything.
    def save_fig(fig, n, name):
        """Save figure as R_<NN>_<name>.pdf. `n` is the fixed figure id: an int (12 -> R_12)
        or a string for split figures ('12a' -> R_12a)."""
        label = f'{n:02d}' if isinstance(n, int) else str(n)
        stem = f"R_{label}_{name}"
        fig.savefig(IMG/f'{stem}.pdf', bbox_inches='tight')
        print(f'saved  images/{stem}.pdf')

    def load_signal(fname=SIG_REF):
        """Load a validated monthly dispersion-signal series (compounded), indexed by month-end date."""
        s = pd.read_parquet(SIG/fname).set_index('date')['rho'].sort_index()
        s.index = pd.to_datetime(s.index) + pd.offsets.MonthEnd(0)
        return s

    def load_or_compute(path, compute_fn, columns=None):
        """check -> load -> else compute -> save. Returns a DataFrame.

        `compute_fn` must return a pandas DataFrame; it is only called on a cache miss.
        """
        path = Path(path)
        if path.exists():
            print(f'loaded cache  {path.relative_to(ROOT)}')
            return pd.read_parquet(path)
        df = compute_fn()
        if columns is not None:
            df = df[columns]
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_suffix(path.suffix + '.tmp')
        df.to_parquet(tmp, index=False)
        tmp.replace(path)
        print(f'computed+saved {path.relative_to(ROOT)} ({len(df)} rows)')
        return df

    assert SIG.exists(),   f'missing signals dir: {SIG}'
    assert PANEL.exists(), f'missing PIT panel: {PANEL}'
    print('setup OK | reference signal ->', SIG_REF, '| panel ->', PANEL.name, '| output ->', OUT)


## 1. Signal overview

The reference signal $\widehat{\rho}_{\tau}$ over the full sample, in its square-root scale $\sqrt{\widehat{\rho}_{\tau}}$ (the scale that enters the ambiguity radius). The expanding mean used for the radius normalisation is overlaid; shaded bands mark the externally dated stress episodes.

In [ ]:
if RUN_MODE == 'full':
    # Figure R_01 — reference dispersion signal with stress episodes
    rho = load_signal()                           # V_uni: big caps, W=36, uniform weights (reference spec)
    sqrt_rho = np.sqrt(rho)
    exp_mean = sqrt_rho.expanding().mean()        # causal expanding mean (radius normaliser)

    fig, ax = plt.subplots(figsize=(10, 4.2))
    for _, pk in CRISES:
        ax.axvspan(pd.Timestamp(pk)-pd.DateOffset(months=3),
                   pd.Timestamp(pk)+pd.DateOffset(months=3),
                   color=_GREY, alpha=0.12, lw=0)

    ax.plot(sqrt_rho.index, sqrt_rho.values, color=_NAVY, lw=1.3,
            label=r'$\sqrt{\widehat{\rho}^{\mathrm{hyb}}_{\tau}}$  ($W=36$)')
    ax.plot(exp_mean.index, exp_mean.values, color=_RUST, lw=1.4, ls='--',
            label=r'$\overline{\sqrt{\widehat{\rho}^{\mathrm{hyb}}}}_{\tau}\,$  (expanding mean)' )
    # crisis labels: horizontal, just above x-axis, below each shaded band
    y0, y1 = ax.get_ylim()
    ax.set_ylim(y0, y1)
    for name, pk in CRISES:
        ax.text(pd.Timestamp(pk), y0 + 0.02*(y1-y0), name,
                ha='center', va='bottom', fontsize=8, color=_GREY)

    ax.set_xlabel('Date'); ax.set_ylabel('Dispersion signal')
    ax.set_title('Multi-frequency dispersion signal and stress episodes (1995–2025)')
    ax.legend(loc='upper right')
    ax.margins(x=0.01)
    fig.tight_layout()
    save_fig(fig, 1, 'signal_overview')
    plt.show()


## 2. Event study: reaction to stress episodes

For each externally dated peak $\tau_c$, the standardized signal $z_\tau = (\sqrt{\hat{\rho}_\tau}-\mu)/\sigma$ is shown on a symmetric $\pm 18$-month window centered on the peak. A consistent rise toward the peak indicates that the signal tracks distributional stress dated independently of it.

In [ ]:
if RUN_MODE == 'full':
    # Figure R_02 — event study: standardized signal around each stress peak
    H = 18  # months on each side of the peak

    z = (np.sqrt(rho) - np.sqrt(rho).mean()) / np.sqrt(rho).std()

    fig, axes = plt.subplots(2, 3, figsize=(11, 6))
    for ax, (name, pk) in zip(axes.flat, CRISES):
        pk = pd.Timestamp(pk) + pd.offsets.MonthEnd(0)
        win = z.loc[pk - pd.DateOffset(months=H): pk + pd.DateOffset(months=H)]
        months = ((win.index.year - pk.year) * 12 + (win.index.month - pk.month))
        ax.axhline(0, color=_GREY, lw=0.6, ls=':')
        ax.axvline(0, color=_RUST, lw=1.0, ls='--')
        ax.plot(months, win.values, color=_NAVY, lw=1.5)
        ax.set_title(f'{name}  ({pk.strftime("%Y-%m")})', fontsize=11)
        ax.set_xlim(-H, H)
        ax.set_xticks([-12, 0, 12])
        ax.margins(y=0.10)

    # shared labels + clean the unused 6th panel
    for ax in axes[1, :]:
        ax.set_xlabel('Months from peak')
    for ax in axes[:, 0]:
        ax.set_ylabel(r'$z_\tau$')
    axes.flat[-1].axis('off')
    axes.flat[-1].text(0.5, 0.5,
        'Standardized signal\n$z_\\tau$ on $[\\tau_c-18,\\ \\tau_c+18]$\n'
        'dashed line: crisis peak',
        ha='center', va='center', fontsize=10, color=_GREY,
        transform=axes.flat[-1].transAxes)

    fig.suptitle('Dispersion signal around externally dated stress episodes', y=0.99)
    fig.tight_layout()
    save_fig(fig, 2, 'event_study')
    plt.show()


## 3. Discriminant power: placebo test

The event study shows that the signal *reacts* to dated stress. We now ask the stricter question: can $\widehat{\rho}_{\tau}$ tell a **genuine cross-frequency misalignment** apart from the finite-sample noise that any estimator produces on a single window?

We use a controlled placebo on the reference universe (big-cap PIT universe, $W=36$ months, uniform $\lambda_k=1/3$):

- **$H_0$ (no misalignment):** the signal is computed on a real window as observed.
- **$H_1$ (injected misalignment):** the *same* window is perturbed — the weekly block is scaled in volatility ($\times 2$) and the monthly block is shifted in mean ($+2\,\widehat{\sigma}_m$). The daily / weekly / monthly blocks then describe genuinely different distributions.

The *same* random seed is used for $H_0$ and $H_1$ on each window, so only the perturbation differs. Across $B$ windows we report three diagnostics: **Cohen's $d$** (standardized separation), the **detection power** at the $H_0$ 95th-percentile threshold, and the **ROC AUC** (threshold-free discrimination).

In [ ]:
if RUN_MODE == 'full':
    # --- B3 estimator socle: compounded, PIT-aligned rho (same construction as notebook 05) ---
    import polars as pl
    import ot
    import gc
    import hashlib
    from sklearn.cluster import KMeans

    # Immutable estimator constants (reference spec), matching 05.
    N_PROJ, N_Q     = 200, 200
    M_PLAC          = 50
    SEED            = 42
    BARY_MAX_ITER   = 30
    BARY_TOL        = 1e-4
    HORIZONS        = np.array([1.0, 5.0, 21.0])   # daily / weekly / monthly
    LOOKBACK_MONTHS = 36

    def seed_for(formation_month, base_seed=20250301, universe='big_caps'):
        """Deterministic per-window seed (common random numbers), same rule as 05."""
        key = f'{base_seed}|{universe}|{pd.Timestamp(formation_month):%Y-%m}'
        return int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)

    # ---- estimators, taken verbatim from notebook 05 (single source of truth) ----
    def _initial_support(arrays, M_atoms):
        pool = np.vstack(arrays)
        sample_weight = np.concatenate([np.full(len(x), 1.0/(3.0*len(x))) for x in arrays])
        model = KMeans(n_clusters=M_atoms, init='k-means++', n_init=1, random_state=0)
        model.fit(pool, sample_weight=sample_weight)
        return model.cluster_centers_

    def _free_support_barycenter(arrays, M_atoms):
        measures_weights = [np.full(len(x), 1.0/len(x)) for x in arrays]
        return ot.lp.free_support_barycenter(
            arrays, measures_weights, X_init=_initial_support(arrays, M_atoms),
            b=np.full(M_atoms, 1.0/M_atoms), weights=np.ones(3)/3.0,
            numItermax=BARY_MAX_ITER, stopThr=BARY_TOL)

    def _sliced_dispersion(arrays, support, lam, seed):
        rng = np.random.default_rng(seed)
        q = np.linspace(0.0, 1.0, N_Q)
        dimension = arrays[0].shape[1]
        total = 0.0
        for _ in range(N_PROJ):
            direction = rng.standard_normal(dimension); direction /= np.linalg.norm(direction)
            bary_q = np.quantile(support @ direction, q)
            total += sum(lam[k]*np.mean((np.quantile(x @ direction, q) - bary_q)**2)
                         for k, x in enumerate(arrays))
        return float(total / N_PROJ)

    def rho_sw(daily, weekly, monthly, M=M_PLAC, lam=None, seed=0):
        """Sliced-Wasserstein dispersion on three ALREADY-SCALED numpy matrices (uniform lambda by default)."""
        if lam is None:
            lam = np.ones(3)/3.0
        arrays = [np.asarray(daily, float), np.asarray(weekly, float), np.asarray(monthly, float)]
        support = _free_support_barycenter(arrays, M)
        return _sliced_dispersion(arrays, support, lam, seed)

    # ---- PIT panel + compounded, aligned windows (no imputation, no per-frequency cleaning) ----
    def _load_pit_pivot():
        """Daily-return wide matrix (index=date, columns=PERMNO) restricted to formation members,
        plus the ordered list of formation months and their formation-close dates."""
        lf = pl.scan_parquet(PANEL).filter(pl.col('is_formation_member') == 1)
        members = (lf.select([
                       pl.col('formation_member_month').alias('formation_month'),
                       'PERMNO', pl.col('DlyCalDt').alias('formation_date')])
                     .collect(engine='streaming').to_pandas())
        members['formation_month'] = pd.to_datetime(members['formation_month'])
        members['formation_date']  = pd.to_datetime(members['formation_date'])
        memberships = {pd.Timestamp(m): g['PERMNO'].astype(int).tolist()
                       for m, g in members.groupby('formation_month', sort=True)}
        formation_date = members.groupby('formation_month')['formation_date'].first().to_dict()

        daily = (pl.read_parquet(PANEL, columns=['PERMNO', 'DlyCalDt', 'DlyRet'])
                   .unique(subset=['PERMNO', 'DlyCalDt'], keep='first'))
        pivot = (daily.pivot(on='PERMNO', index='DlyCalDt', values='DlyRet')
                      .sort('DlyCalDt').to_pandas().set_index('DlyCalDt'))
        pivot.index = pd.to_datetime(pivot.index)
        pivot.columns = pivot.columns.astype(int)
        return pivot, memberships, {pd.Timestamp(k): pd.Timestamp(v) for k, v in formation_date.items()}

    def pit_matrices(pivot, formation_month, memberships, formation_date, lookback=LOOKBACK_MONTHS):
        """Return SCALED (sqrt-h) daily/weekly/monthly numpy matrices for the 100 PIT members.
        Weekly/monthly are geometrically COMPOUNDED. Asserts a full matrix (no imputation)."""
        fm = pd.Timestamp(formation_month)
        assets = memberships[fm]
        fdate  = pd.Timestamp(formation_date[fm])
        start  = fm - pd.DateOffset(months=lookback-1)

        d = pivot.loc[(pivot.index >= pd.Timestamp(start)) & (pivot.index <= fdate), assets]
        assert d.notna().all().all(), f'{fm:%Y-%m}: PIT daily matrix is not full (would require imputation).'
        w = (1.0 + d).resample('W-FRI').prod(min_count=1) - 1.0
        m = (1.0 + d).resample('ME').prod(min_count=1) - 1.0
        assert w.notna().all().all() and m.notna().all().all()
        Xd, Xw, Xm = (d.to_numpy(float)/HORIZONS[0]**0.5,
                      w.to_numpy(float)/HORIZONS[1]**0.5,
                      m.to_numpy(float)/HORIZONS[2]**0.5)
        return Xd, Xw, Xm

    # formation months usable as 36-month window ends
    PIVOT, MEMB, FDATE = _load_pit_pivot()
    _all_months = sorted(MEMB)
    WINDOWS = [fm for fm in _all_months
               if (fm - pd.DateOffset(months=LOOKBACK_MONTHS-1)) >= PIVOT.index.min()]
    print(f'PIT panel: {len(_all_months)} formation months | {len(WINDOWS)} usable 36-month windows '
          f'({WINDOWS[0]:%Y-%m} -> {WINDOWS[-1]:%Y-%m})')


In [ ]:
if RUN_MODE == 'full':
    # Figure R_03 — placebo: discriminant power of the signal (H0 vs H1)
    # H0 = observed PIT window; H1 = same window with injected misalignment (vol x2 weekly, drift on monthly).
    from sklearn.metrics import roc_curve, auc

    B_PLAC, VOL_SHOCK, DRIFT_SH = 300, 2.0, 2.0
    PLAC_CACHE = CACHE_PLAC/'placebo_h0h1.parquet'

    def _compute_placebo():
        rng = np.random.default_rng(SEED)
        picks = rng.choice(len(WINDOWS), size=min(B_PLAC, len(WINDOWS)), replace=False)
        print(f'Placebo | B={len(picks)} | M={M_PLAC} | vol x{VOL_SHOCK} | drift {DRIFT_SH}sigma | lambda=1/3')
        rows = []
        for k, idx in enumerate(picks):
            fm = WINDOWS[idx]
            Xd, Xw, Xm = pit_matrices(PIVOT, fm, MEMB, FDATE)
            s = seed_for(fm)
            h0 = rho_sw(Xd, Xw, Xm, seed=s)
            sig_m = Xm.std(axis=0)
            h1 = rho_sw(Xd, Xw*VOL_SHOCK, Xm + DRIFT_SH*sig_m[None, :], seed=s)
            rows.append({'formation_month': fm, 'h0': h0, 'h1': h1})
            gc.collect()
            if (k+1) % 50 == 0:
                print(f'  [{k+1}/{len(picks)}] {fm:%Y-%m}')
        return pd.DataFrame(rows)

    plac = load_or_compute(PLAC_CACHE, _compute_placebo)
    h0, h1 = plac['h0'].to_numpy(), plac['h1'].to_numpy()

    # --- diagnostics ---
    cohen_d = (h1.mean() - h0.mean()) / np.sqrt((h0.var(ddof=1) + h1.var(ddof=1)) / 2)
    thr     = np.percentile(h0, 95)
    power   = (h1 > thr).mean()
    y_true  = np.r_[np.zeros(len(h0)), np.ones(len(h1))]
    y_score = np.r_[h0, h1]
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    ratio   = h1.mean() / h0.mean()

    # --- figure: distributions + ROC ---
    fig, (axd, axr) = plt.subplots(1, 2, figsize=(11, 4.4))
    bins = np.linspace(0, max(h0.max(), h1.max()) * 1.05, 38)
    axd.hist(h0, bins=bins, color=_NAVY, alpha=0.55, label=r'$H_0$  (no misalignment)')
    axd.hist(h1, bins=bins, color=_RUST, alpha=0.55, label=r'$H_1$  (injected misalignment)')
    axd.axvline(thr, color=_GREY, lw=1.2, ls='--', label=r'$H_0$ 95th pct.')
    axd.set_xlabel(r'$\widehat{\rho}^{\mathrm{hyb}}$'); axd.set_ylabel('Count')
    axd.set_title('Signal under $H_0$ vs $H_1$'); axd.legend(loc='upper right')

    axr.plot(fpr, tpr, color=_GREEN, lw=1.8)
    axr.plot([0, 1], [0, 1], color=_GREY, lw=0.8, ls=':')
    axr.set_xlabel('False positive rate'); axr.set_ylabel('True positive rate')
    axr.set_title(f'ROC curve  (AUC = {roc_auc:.3f})')
    axr.set_xlim(-0.02, 1.02); axr.set_ylim(-0.02, 1.02)
    axr.text(0.96, 0.06,
             f"Cohen's $d$ = {cohen_d:.2f}\nPower = {power*100:.0f}%\nRatio $H_1/H_0$ = {ratio:.1f}$\\times$",
             transform=axr.transAxes, ha='right', va='bottom', fontsize=9.5,
             bbox=dict(boxstyle='round', fc='white', ec=_GREY, alpha=0.9))

    fig.suptitle('Discriminant power of the dispersion signal (placebo test)', y=0.99)
    fig.tight_layout()
    save_fig(fig, 3, 'placebo_discrimination')
    plt.show()
    print(f"B={len(h0)} | Cohen d={cohen_d:.3f} | power(p95)={power*100:.1f}% | AUC={roc_auc:.4f} | ratio={ratio:.1f}x")

    # --- export LaTeX table ---
    tab = (
        "\\begin{tabular}{lc}\n\\toprule\n"
        "Diagnostic & Value \\\\\n\\midrule\n"
        f"Replications $B$ & {len(h0)} \\\\\n"
        f"$\\widehat{{\\rho}}$ under $H_0$ (mean) & {h0.mean():.2e} \\\\\n"
        f"$\\widehat{{\\rho}}$ under $H_1$ (mean) & {h1.mean():.2e} \\\\\n"
        f"Ratio $H_1/H_0$ & {ratio:.1f}$\\times$ \\\\\n"
        f"Cohen's $d$ & {cohen_d:.2f} \\\\\n"
        f"Detection power (95\\% thr.) & {power*100:.1f}\\% \\\\\n"
        f"ROC AUC & {roc_auc:.3f} \\\\\n"
        "\\bottomrule\n\\end{tabular}\n"
    )
    (TAB/'R_13_placebo_discrimination.tex').write_text(tab)
    print('saved  tables/R_13_placebo_discrimination.tex')


### 3.1. Power as a function of misalignment intensity

A single perturbation gives a single point. To show that the discrimination is *graded* rather than an artefact of one oversized shock, we trace the **detection power** as the misalignment intensity is swept. Two axes are varied independently, holding the other at its null value:

- **Volatility misalignment:** the weekly block is scaled by a factor in $\{1,\,1.25,\,1.5,\,2,\,3\}$ (factor $1$ = no shock).
- **Drift misalignment:** the monthly block is shifted by $\{0,\,0.5,\,1,\,2,\,3\}\,\widehat{\sigma}_m$.

Power is measured against the same $H_0$ 95th-percentile threshold. A curve that climbs smoothly from the nominal $5\%$ size (no shock) toward $100\%$ confirms that the signal responds monotonically to the *amount* of misalignment, not merely to its presence.

In [ ]:
if RUN_MODE == 'full':
    # Figure R_04 — power curves: detection power vs misalignment intensity
    # Reuses the PIT windows; caches the raw rho grids so the sweep is not recomputed.
    B_SENS = 300
    POW_CACHE = CACHE_POW/'power_curves.parquet'
    vol_grid   = [1.0, 1.25, 1.5, 2.0, 3.0]
    drift_grid = [0.0, 0.5, 1.0, 2.0, 3.0]

    def _valid_windows(picks):
        out = []
        for idx in picks:
            fm = WINDOWS[idx]
            Xd, Xw, Xm = pit_matrices(PIVOT, fm, MEMB, FDATE)
            out.append((fm, Xd, Xw, Xm))
        return out

    def _compute_power():
        rng = np.random.default_rng(SEED + 1)
        picks = rng.choice(len(WINDOWS), size=min(B_SENS, len(WINDOWS)), replace=False)
        W = _valid_windows(picks)
        rho0 = np.array([rho_sw(Xd, Xw, Xm, seed=seed_for(fm)) for fm, Xd, Xw, Xm in W])
        thr  = np.percentile(rho0, 95)
        rows = [{'kind': 'threshold', 'x': 0.0, 'power': thr}]
        for vs in vol_grid:
            r = np.array([rho_sw(Xd, Xw*vs, Xm, seed=seed_for(fm)) for fm, Xd, Xw, Xm in W])
            rows.append({'kind': 'vol', 'x': vs, 'power': (r > thr).mean()*100})
        for ds in drift_grid:
            r = np.array([rho_sw(Xd, Xw, Xm + ds*Xm.std(axis=0)[None, :], seed=seed_for(fm))
                          for fm, Xd, Xw, Xm in W])
            rows.append({'kind': 'drift', 'x': ds, 'power': (r > thr).mean()*100})
        return pd.DataFrame(rows)

    pw = load_or_compute(POW_CACHE, _compute_power)
    pow_vol   = pw.loc[pw['kind'].eq('vol')].sort_values('x')['power'].tolist()
    pow_drift = pw.loc[pw['kind'].eq('drift')].sort_values('x')['power'].tolist()

    # --- figure ---
    fig, (axv, axdr) = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, x, y, xlab, ttl, col in [
        (axv, vol_grid, pow_vol, r'weekly volatility factor', 'Volatility misalignment', _RUST),
        (axdr, drift_grid, pow_drift, r'monthly drift shift ($\times\,\hat{\sigma}_m$)',
         'Drift misalignment', _NAVY)]:
        ax.plot(x, y, 'o-', color=col, lw=1.8, ms=6)
        ax.axhline(80, color=_GREY, lw=0.8, ls='--', label='80% power')
        ax.axhline(5, color=_GREY, lw=0.8, ls=':', label='5% (size)')
        ax.set_xlabel(xlab); ax.set_ylabel('Detection power (%)')
        ax.set_title(ttl); ax.set_ylim(-3, 105)
        ax.legend(loc='lower right')

    fig.suptitle('Detection power vs misalignment intensity (placebo)', y=0.99)
    fig.tight_layout()
    save_fig(fig, 4, 'power_curves')
    plt.show()
    print('vol  :', dict(zip(vol_grid, [round(p, 1) for p in pow_vol])))
    print('drift:', dict(zip(drift_grid, [round(p, 1) for p in pow_drift])))

    # --- export LaTeX table ---
    rows = "".join(
        f"{vs:g} & {pv:.1f}\\% & {ds:g} & {pd_:.1f}\\% \\\\\n"
        for vs, pv, ds, pd_ in zip(vol_grid, pow_vol, drift_grid, pow_drift))
    tab = (
        "\\begin{tabular}{cccc}\n\\toprule\n"
        "Vol. factor & Power & Drift ($\\hat{\\sigma}_m$) & Power \\\\\n\\midrule\n"
        + rows +
        "\\bottomrule\n\\end{tabular}\n"
    )
    (TAB/'R_14_power_curves.tex').write_text(tab)
    print('saved  tables/R_14_power_curves.tex')


### 3.2. ROC and AUC as the misalignment intensity grows

The power curves use a single threshold. The ROC curve sweeps *all* thresholds: $\mathrm{TPR}(q)$ against $\mathrm{FPR}(q)$. Its area (AUC) equals $\mathbb{P}(\widehat{\rho}^{(1)} > \widehat{\rho}^{(0)})$ — the probability that a perturbed window scores higher than an unperturbed one ($0.5$ = random, $1$ = perfect separation).

Rather than the AUC at the reference shock alone, we trace one ROC curve per intensity along each axis — weekly-volatility scaling $\kappa$ and monthly-drift shift $\xi$ — with a bootstrap confidence interval on each AUC. As the shock grows, the curve lifts off the diagonal and the AUC climbs from $0.5$ toward $1$: discrimination is *built up* gradually, not asserted at a saturated point.

In [ ]:
if RUN_MODE == 'full':
    # Figure R_05 — ROC family as shock intensity grows (kappa | xi), AUC + bootstrap CI
    from sklearn.metrics import roc_curve, auc

    ROC_CACHE = CACHE_POW/'roc_intensity.parquet'
    KAPPA = [1.25, 1.5, 2.0, 3.0]      # weekly-vol scaling (xi=0)
    XI    = [0.5, 1.0, 2.0, 3.0]       # monthly-drift shift (kappa=1)
    N_BOOT = 500

    def _roc_windows():
        rng = np.random.default_rng(SEED + 1)
        picks = rng.choice(len(WINDOWS), size=min(B_SENS, len(WINDOWS)), replace=False)
        return [(WINDOWS[i], *pit_matrices(PIVOT, WINDOWS[i], MEMB, FDATE)) for i in picks]

    def _compute_roc():
        W = _roc_windows()
        rows = []
        rho0 = np.array([rho_sw(Xd, Xw, Xm, seed=seed_for(fm)) for fm, Xd, Xw, Xm in W])
        for j, r in enumerate(rho0):
            rows.append({'grid': 'H0', 'val': 0.0, 'win': j, 'rho': r})
        for kappa in KAPPA:
            r = [rho_sw(Xd, Xw*kappa, Xm, seed=seed_for(fm)) for fm, Xd, Xw, Xm in W]
            rows += [{'grid': 'kappa', 'val': kappa, 'win': j, 'rho': v} for j, v in enumerate(r)]
        for xi in XI:
            r = [rho_sw(Xd, Xw, Xm + xi*Xm.std(axis=0)[None, :], seed=seed_for(fm)) for fm, Xd, Xw, Xm in W]
            rows += [{'grid': 'xi', 'val': xi, 'win': j, 'rho': v} for j, v in enumerate(r)]
        return pd.DataFrame(rows)

    roc_df = load_or_compute(ROC_CACHE, _compute_roc)
    _rho_H0 = roc_df.loc[roc_df['grid'].eq('H0')].sort_values('win')['rho'].to_numpy()
    print(f'fenetres valides = {len(_rho_H0)} | bootstrap reps = {N_BOOT}')

    def _series(grid, val):
        return roc_df.loc[roc_df['grid'].eq(grid) & roc_df['val'].eq(val)].sort_values('win')['rho'].to_numpy()

    def _auc_ci(h0, h1, B=N_BOOT, seed=SEED):
        if len(h0) != len(h1):
            raise ValueError('Paired AUC bootstrap requires one H1 value per H0 window')
        y = np.r_[np.zeros(len(h0)), np.ones(len(h1))]; s = np.r_[h0, h1]
        fpr, tpr, _ = roc_curve(y, s); a = auc(fpr, tpr)
        rng = np.random.default_rng(seed); n = len(h0); boot = []
        for _ in range(B):
            idx = rng.integers(0, n, n)  # preserve the H0/H1 formation-window pairing
            yy = np.r_[np.zeros(n), np.ones(n)]; ss = np.r_[h0[idx], h1[idx]]
            f, t, _ = roc_curve(yy, ss); boot.append(auc(f, t))
        return fpr, tpr, a, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

    fig, (axk, axx) = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
    _cmap = plt.cm.viridis(np.linspace(0.15, 0.9, 4))
    for ax, grid, kind in [(axk, KAPPA, 'kappa'), (axx, XI, 'xi')]:
        ax.plot([0, 1], [0, 1], color=_GREY, lw=0.8, ls=':')
        for val, c in zip(grid, _cmap):
            h1 = _series(kind, val)
            fpr, tpr, a, lo, hi = _auc_ci(_rho_H0, h1)
            lbl = (rf'$\kappa={val:g}$' if kind == 'kappa' else rf'$\xi={val:g}$') + f'  (AUC {a:.2f} [{lo:.2f},{hi:.2f}])'
            ax.plot(fpr, tpr, color=c, lw=1.6, label=lbl)
        ax.set_xlabel('False positive rate')
        ax.set_title('Weekly volatility $\\kappa$' if kind == 'kappa' else 'Monthly drift $\\xi$')
        ax.legend(loc='lower right', fontsize=8)
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
    axk.set_ylabel('True positive rate')

    fig.suptitle('ROC curves built up across misalignment intensity', y=0.99)
    fig.tight_layout()
    save_fig(fig, 5, 'roc_family')
    plt.show()

    # export LaTeX: AUC + CI per intensity, both axes
    def _rows(grid, kind):
        return [(v, *_auc_ci(_rho_H0, _series(kind, v))[2:]) for v in grid]
    rk, rx = _rows(KAPPA, 'kappa'), _rows(XI, 'xi')
    body = ''.join(f"${k:g}$ & {ak:.3f} & $[{lk:.3f},{hk:.3f}]$ & ${x:g}$ & {ax_:.3f} & $[{lx:.3f},{hx:.3f}]$ \\\\\n"
                   for (k,ak,lk,hk),(x,ax_,lx,hx) in zip(rk, rx))
    tab = ("\\begin{tabular}{cccccc}\n\\toprule\n"
           "\\multicolumn{3}{c}{Weekly vol. $\\kappa$} & \\multicolumn{3}{c}{Monthly drift $\\xi$} \\\\\n"
           "\\cmidrule(lr){1-3}\\cmidrule(lr){4-6}\n"
           "$\\kappa$ & AUC & 95\\% CI & $\\xi$ & AUC & 95\\% CI \\\\\n\\midrule\n"
           + body + "\\bottomrule\n\\end{tabular}\n")
    (TAB/'R_15_roc_family.tex').write_text(tab)
    print('saved  tables/R_15_roc_family.tex')

    # --- clean console recap of the ROC family (reuses rk / rx, no recompute) ---
    _W = 50
    print('\n' + '─' * _W)
    print('  ROC — discrimination vs misalignment intensity')
    print('─' * _W)
    print(f'  {"Weekly volatility κ":<24}{"Monthly drift ξ"}')
    for (k, ak, lk, hk), (x, ax_, lx, hx) in zip(rk, rx):
        left  = f'κ={k:<4g} AUC {ak:.2f} [{lk:.2f},{hk:.2f}]'
        right = f'ξ={x:<4g} AUC {ax_:.2f} [{lx:.2f},{hx:.2f}]'
        print(f'  {left:<24}{right}')
    print('─' * _W)


## 4. Robustness to construction choices

The placebo establishes that the signal carries information; we now check that this information does not hinge on the *numerical choices* made when building it. Each robustness series is the **validated, compounded** variant published under `data/signals/` — the reference universe is held fixed (big-cap PIT universe, $W=36$ months) and exactly one construction lever is perturbed at a time. If the signal chronology is stable across a lever, that lever is not driving the result.

The correlation table reports the overall correlation with the reference and three sub-period correlations (1995–2004, 2004–2015, 2015–2025). These sub-periods **overlap at their boundary years** (2004 and 2015 each appear in two columns); they are read as descriptive stability checks, not as three independent tests.

In [ ]:
if RUN_MODE == 'full':
    # --- B4 socle: robustness variants LOADED from data/signals (compounded, validated by 05/tests-02) ---
    ref = load_signal('V_uni.parquet')     # big caps, W=36, uniform (reference)

    SENS = {
        'cardinality': {
            r'$M=50$ (ref.)':               ref,
            r'$M=36$':                      load_signal('U_M36.parquet'),
            r'$M=72$':                      load_signal('U_M72.parquet'),
        },
        'weights': {
            r'uniform (ref.)':              ref,
            r'$\lambda \propto T_k$':       load_signal('U_lamTk.parquet'),
            r'$\lambda \propto \log T_k$':  load_signal('U_lamlog.parquet'),
        },
        'scaling': {
            r'$\sqrt{h_k}$ (ref.)':         ref,
            r'realized vol.':               load_signal('U_volscale.parquet'),
        },
        'scaling_exponent': {
            r'$h_k^{0.5}$ (ref.)':          ref,
            r'$h_k^{0.4}$':                 load_signal('U_H04.parquet'),
            r'$h_k^{0.6}$':                 load_signal('U_H06.parquet'),
        },
        'distance': {
            r'Sliced-Wasserstein (ref.)':   ref,
            r'discrete $W_2$ (no slicing)': load_signal('U_exact.parquet'),
        },
        'barycenter': {
            r'$\mathbb{R}^N$ free-supp. (ref.)': ref,
            r'1-D quantile':                load_signal('U_bary1d.parquet'),
        },
    }

    # --- correlation table vs reference (overall + sub-periods), all variants ---
    PERIODS = [('1995-2004', '1995-01', '2004-12'),
               ('2004-2015', '2004-01', '2015-12'),
               ('2015-2025', '2015-01', '2025-12')]
    rows = []
    for axis, group in SENS.items():
        for name, s in group.items():
            if 'ref.' in name:
                continue
            df = pd.DataFrame({'ref': ref, 'v': s}).dropna()
            subs = [df.loc[t0:t1]['ref'].corr(df.loc[t0:t1]['v']) if len(df.loc[t0:t1]) > 2 else np.nan
                    for _, t0, t1 in PERIODS]
            rows.append((axis, name, df['ref'].corr(df['v']), *subs))
    corr_tab = pd.DataFrame(rows, columns=['Axis', 'Variant', 'Overall', *[p[0] for p in PERIODS]])

    # console: a NORMAL readable table (never print LaTeX)
    print('Correlation with the reference signal (compounded):')
    print(corr_tab.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

    # --- export LaTeX correlation table (grouped by axis, American-English decimals) ---
    AXIS_LABEL = {
        'cardinality':       r'Cardinality \(M\)',
        'weights':           r'Frequency weights \(\lambda_k\)',
        'scaling':           r'Scaling',
        'scaling_exponent':  r'Scaling exponent \(H\)',
        'distance':          r'Distance',
        'barycenter':        r'Barycenter',
    }
    def _fmt(x):
        return '\\(' + (f'{x:.3f}' if pd.notna(x) else '\\text{--}') + '\\)'

    lines, first_axis = [], True
    for axis in corr_tab['Axis'].unique():
        if not first_axis:
            lines.append(r'      \addlinespace')
        first_axis = False
        sub = corr_tab[corr_tab['Axis'] == axis]
        for j, r in enumerate(sub.itertuples()):
            lev = AXIS_LABEL.get(axis, axis) if j == 0 else ''
            lines.append(
                f'      {lev} & {r.Variant} & {_fmt(r.Overall)} & '
                f'{_fmt(getattr(r, "_4"))} & {_fmt(getattr(r, "_5"))} & {_fmt(getattr(r, "_6"))} \\\\')
    body = '\n'.join(lines)
    tab = (
        r'    \begin{tabular}{llcccc}' '\n'
        r'      \toprule' '\n'
        r'      & & \multicolumn{4}{c}{Correlation with the reference} \\' '\n'
        r'      \cmidrule(lr){3-6}' '\n'
        r'      Lever & Variant & Overall & 1995--2004 & 2004--2015 & 2015--2025 \\' '\n'
        r'      \midrule' '\n'
        + body + '\n'
        r'      \bottomrule' '\n'
        r'    \end{tabular}%' '\n'
    )
    (TAB/'R_16_sensitivity_corr.tex').write_text(tab)
    print('\nsaved  tables/R_16_sensitivity_corr.tex')

    # --- shared overlay plotters (reference navy/thick vs vivid variants) ---
    _VIVID = [_NAVY, _RUST, _GREEN, '#e08214']

    def _zs(s):
        return (np.sqrt(s) - np.sqrt(s).mean()) / np.sqrt(s).std()

    def event_study_overlay(group, n, title, fname, H=18):
        """2x3 event study around CRISES; reference (navy, thick) vs variants (vivid)."""
        fig, axes = plt.subplots(2, 3, figsize=(11, 6), sharey=False)
        for ax, (name, pk) in zip(axes.flat, CRISES):
            pk = pd.Timestamp(pk) + pd.offsets.MonthEnd(0)
            for (lbl, s), c in zip(group.items(), _VIVID):
                is_ref = ('ref.' in lbl)
                z = _zs(s)
                win = z.loc[pk - pd.DateOffset(months=H): pk + pd.DateOffset(months=H)]
                mo = (win.index.year - pk.year)*12 + (win.index.month - pk.month)
                ax.plot(mo, win.values, color=c, lw=2.0 if is_ref else 1.4,
                        zorder=3 if is_ref else 2, alpha=1.0 if is_ref else 0.9)
            ax.axvline(0, color=_GREY, lw=0.9, ls='--')
            ax.axhline(0, color=_GREY, lw=0.6, ls=':')
            ax.set_title(f'{name}  ({pk.strftime("%Y-%m")})', fontsize=11)
            ax.set_xlim(-H, H); ax.set_xticks([-12, 0, 12])
        for ax in axes[1, :]:
            ax.set_xlabel('Months from peak')
        for ax in axes[:, 0]:
            ax.set_ylabel(r'$z_\tau$')
        axes.flat[-1].axis('off')
        handles = [plt.Line2D([], [], color=c, lw=2.0 if 'ref.' in l else 1.4)
                   for l, c in zip(group.keys(), _VIVID)]
        axes.flat[-1].legend(handles, list(group.keys()), loc='center', fontsize=9.5,
                             title=title, title_fontsize=10)
        fig.suptitle(f'Robustness: {title}', y=0.99)
        fig.tight_layout()
        save_fig(fig, n, fname)
        plt.show()

    def timeseries_overlay(group, n, title, fname, show_corr=True, legend_loc='upper left'):
        """Full-sample raw rho (x1e-4) 1995-2025; reference navy (thick) vs vivid variants.
        Raw scale keeps the level differences visible (z-score would hide them)."""
        fig, ax = plt.subplots(figsize=(10, 4.2))
        for _, pk in CRISES:
            ax.axvspan(pd.Timestamp(pk)-pd.DateOffset(months=3),
                       pd.Timestamp(pk)+pd.DateOffset(months=3),
                       color=_GREY, alpha=0.10, lw=0)
        for (lbl, s), c in zip(group.items(), _VIVID):
            is_ref = ('ref.' in lbl)
            if is_ref or not show_corr:
                cc = ''
            else:
                cc = f"  (corr {pd.DataFrame({'a':ref,'b':s}).dropna().corr().iloc[0,1]:.3f})"
            ax.plot(s.index, s.values*1e4, color=c, lw=2.0 if is_ref else 1.3,
                    zorder=3 if is_ref else 2, alpha=1.0 if is_ref else 0.85, label=lbl + cc)
        ax.set_xlabel('Date'); ax.set_ylabel(r'$\widehat{\rho}^{\mathrm{hyb}}_{\tau}\ \times 10^{-4}$')
        ax.set_title(f'Robustness: {title}  (1995-2025)')
        ax.legend(loc=legend_loc, ncol=1)
        ax.margins(x=0.01)
        fig.tight_layout()
        save_fig(fig, n, fname)
        plt.show()

    print('\nSENS axes ready:', list(SENS.keys()))


### 4.1. Barycenter cardinality $M$

In [ ]:
if RUN_MODE == 'full':
    # Figure R_06 — robustness to barycenter cardinality M (36 / 50 / 72)
    # Full-sample series: the three curves stay glued across 1995-2025.
    timeseries_overlay(SENS['cardinality'], 6, 'barycenter cardinality $M$', 'robust_M',
                       legend_loc='upper right')


### 4.2. Frequency weights $\lambda_k$

In [ ]:
if RUN_MODE == 'full':
    # Figure R_07 — robustness to frequency weighting (uniform / T_k / log T_k)
    event_study_overlay(SENS['weights'], 7, r'frequency weights $\lambda_k$', 'robust_lambda')


### 4.3. Inter-frequency scaling

Unlike the other levers, replacing the $\sqrt{h_k}$ scaling by per-frequency realized
volatility is **not** a small perturbation of the reference: its correlation with the
reference signal is low (around $0.2$ overall, and negative in the early sub-period).
This is expected — dividing each frequency block by its own realized volatility removes
the very cross-horizon scale differences the signal is built to measure, so it defines a
**different economic object** rather than a robustness variant. It is reported here for
completeness and should be read as such, not as evidence of instability of the reference.

In [ ]:
if RUN_MODE == 'full':
    # Figure R_08 — robustness to inter-frequency scaling (sqrt(h) vs realized vol.)
    event_study_overlay(SENS['scaling'], 8, 'inter-frequency scaling', 'robust_scaling')


### 4.4. Transport distance

In [ ]:
if RUN_MODE == 'full':
    # Figure R_09 — robustness to transport distance (sliced vs exact W2)
    event_study_overlay(SENS['distance'], 9, 'transport distance', 'robust_distance')


### 4.5. Monte-Carlo convergence of the sliced estimator

The sliced approximation relies on $L$ random projections: it is a Monte-Carlo
estimator whose variance decreases in $1/\sqrt{L}$. We measure its stability through
the coefficient of variation $\mathrm{CV} = \sigma/\mu$ of the signal $\rho$
recomputed under several draws, as a function of $L$, on two contrasting regimes:
a stress window (GFC, ending 2009-09) and a calm window (2013–2016, ending 2016-06).
The barycentre support is fixed once per regime and the random seed is held across
$L$ values, so the curve isolates the effect of the projection count alone.

In [ ]:
if RUN_MODE == 'full':
    # Figure R_10 — Monte-Carlo convergence of the sliced estimator (CV of rho vs L, two regimes)
    # Only recompute in part 4: caches to cache/monte_carlo/. Depends on B3 socle (PIVOT, pit_matrices, rho_sw).
    from matplotlib.ticker import FixedLocator, FixedFormatter

    MC_PATH = CACHE_MC/'mc_convergence.parquet'
    L_GRID  = [10, 20, 50, 100, 200, 500, 1000]
    N_REPS  = 30
    # 36-month windows ENDING at the labelled month, so the labels match the windows:
    #   GFC  window ends 2009-09  (~2006-10 -> 2009-09)
    #   calm window ends 2016-06  (~2013-07 -> 2016-06)
    REGIMES = [('Stress (GFC)',      '2009-09'),
               ('Calm (2013-2016)',  '2016-06')]

    def _regime_matrices(month_label):
        """Scaled daily/weekly/monthly matrices for the 36-month window ending at `month_label`."""
        target = pd.Timestamp(month_label)
        target = pd.Timestamp(target.year, target.month, 1)
        fm = min(WINDOWS, key=lambda w: abs((w - target).days))   # snap to an available formation month
        return pit_matrices(PIVOT, fm, MEMB, FDATE), fm

    def _sliced_L(arrays, support, lam, seed, L):
        """Sliced dispersion using EXACTLY L random projections (fixes the Monte-Carlo bug).
        Same seed across L values -> nested common random numbers: the first directions of
        L=50 are reused inside L=100, so the curve isolates the effect of L alone."""
        rng = np.random.default_rng(seed)
        q = np.linspace(0.0, 1.0, N_Q)
        dimension = arrays[0].shape[1]
        total = 0.0
        for _ in range(L):
            direction = rng.standard_normal(dimension); direction /= np.linalg.norm(direction)
            bary_q = np.quantile(support @ direction, q)
            total += sum(lam[k]*np.mean((np.quantile(x @ direction, q) - bary_q)**2)
                         for k, x in enumerate(arrays))
        return float(total / L)

    def _compute_mc():
        rows = []
        lam = np.ones(3)/3.0
        for reg, month_label in REGIMES:
            (Xd, Xw, Xm), fm = _regime_matrices(month_label)
            support = _free_support_barycenter([Xd, Xw, Xm], M_PLAC)   # deterministic: computed ONCE per regime
            print(f'  {reg}: window ends {fm:%Y-%m}')
            for L in L_GRID:
                vals = np.array([_sliced_L([Xd, Xw, Xm], support, lam, seed=k, L=L)
                                 for k in range(N_REPS)])
                rows.append({'regime': reg, 'window_end': f'{fm:%Y-%m}', 'L': L,
                             'mean': vals.mean(), 'std': vals.std(),
                             'cv_pct': vals.std()/vals.mean()*100})
        return pd.DataFrame(rows)

    mc = load_or_compute(MC_PATH, _compute_mc)

    # --- figure: CV(%) vs L, one line per regime, L=200 marked ---
    fig, ax = plt.subplots(figsize=(8.2, 4.2))
    _cols = {'Stress (GFC)': _RUST, 'Calm (2013-2016)': _NAVY}
    for reg in mc['regime'].unique():
        d = mc[mc['regime'] == reg].sort_values('L')
        ax.plot(d['L'], d['cv_pct'], marker='o', ms=5, lw=1.8,
                color=_cols.get(reg, _GREY), label=reg, zorder=3)
    ax.axvline(200, color=_GREY, lw=1.0, ls=':', zorder=2)
    ax.text(200, ax.get_ylim()[1]*0.92, r'$L=200$ (retained)', rotation=90,
            va='top', ha='right', fontsize=8.5, color=_GREY)
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(FixedLocator(L_GRID))
    ax.xaxis.set_major_formatter(FixedFormatter([str(l) for l in L_GRID]))
    ax.set_xlabel('Number of projections $L$ (log scale)')
    ax.set_ylabel(r'Coefficient of variation of $\widehat{\rho}_{\tau,L}$  (%)')
    ax.set_title('Monte-Carlo convergence of the sliced dispersion signal')
    ax.legend(loc='upper right', fontsize=9)
    fig.tight_layout(); save_fig(fig, 10, 'mc_convergence'); plt.show()

    _p200 = mc[mc['L'] == 200].set_index('regime')['cv_pct']
    print('CV at L=200:', {r: f'{_p200[r]:.2f}%' for r in _p200.index})


### 4.6. Barycenter geometry

In [ ]:
if RUN_MODE == 'full':
    # Figure R_11 — robustness to barycenter geometry (R^N free-support vs 1-D quantile)
    event_study_overlay(SENS['barycenter'], 11, 'barycenter geometry', 'robust_barycenter')


### 4.7. Inter-frequency scaling exponent

In [ ]:
if RUN_MODE == 'full':
    # Figures R_12a / R_12b — robustness to the inter-frequency scaling exponent h_k^H (H in {0.4,0.5,0.6})
    # R_12a: full-sample time series (level differences).  R_12b: event study around crises.
    timeseries_overlay(SENS['scaling_exponent'], '12a', 'scaling exponent $h_k^{H}$',
                       'robust_scaling_exponent_ts', show_corr=False)
    event_study_overlay(SENS['scaling_exponent'], '12b', 'scaling exponent $h_k^{H}$',
                        'robust_scaling_exponent')


## 5. Non-redundancy with volatility

A natural objection is that $\sqrt{\widehat{\rho}_{\tau}}$ merely repackages market volatility: cross-horizon dispersion rises in stress, and so do volatility indicators. If the signal were a deterministic function of volatility, it would carry no information of its own.

We measure how much of the signal is explained by two complementary volatility measures: a market **implied volatility** $\mathrm{IV}_\tau$ (forward-looking, the CBOE VIX) and a **realized volatility** $\mathrm{RV}_\tau$ (backward-looking, from the daily returns of the active PIT universe). Crucially, the reference signal is built on a **36-month** rolling window, so comparing it to *monthly* volatility mixes two different time scales. We therefore report the diagnostic at **two matched horizons**:

- **short-term (monthly):** monthly IV and monthly RV, which are reactive;
- **long-horizon (36-month):** a **formation-universe realized volatility** and a 36-month trailing average of the VIX. The realized measure is the tightest possible control: for each formation month it is the annualized volatility of the equal-weight daily return of *that month's exact 100 members* over *their own 36-month window* — the same universe and window used to build $\rho$. It is point-in-time (the composition is known at the formation date and only its prior history is used) and, because members' pre-1995 history is available, it keeps the full 1995–2025 sample with a complete 36-month history at every date. The VIX average is a *matched smoothing window*, not a genuine 36-month implied-volatility horizon — the VIX is always a 30-day expectation, here averaged over 36 months.

An honest reading requires both: a low explanatory power at the monthly horizon can be a mechanical scale mismatch, while the long-horizon regression reveals how much of the signal is genuinely shared with the volatility of the very same assets.

In [ ]:
if RUN_MODE == 'full':
    # 5.a — load signal + IV + RV at two matched horizons, and overlay them (z-score)
    import statsmodels.api as sm
    from statsmodels.tsa.stattools import adfuller

    W_LH = 36       # long-horizon window (months), matching the signal's construction

    # (a) reference signal in sqrt scale (the scale that enters the radius)
    sig = np.sqrt(load_signal())                                   # V_uni (compounded), monthly month-end

    # (b) implied volatility from the DAILY VIX -> monthly mean (short) and 36m MA (long horizon)
    _iv = pd.read_parquet(VIX).set_index('date')['vix']
    _iv.index = pd.to_datetime(_iv.index)
    iv_m  = _iv.resample('ME').mean()                              # monthly IV (reactive)
    iv_lh = iv_m.rolling(W_LH, min_periods=W_LH).mean()            # 36m moving-average of the VIX (smoothed)

    # (c) realized volatility. Two objects at two horizons:
    #   - monthly RV (reactive): std of the equal-weight ACTIVE-universe daily return each month
    #   - formation-universe 36m RV (long horizon, PIT): for each formation month, the annualized std
    #     of the equal-weight daily return of THAT month's 100 members over their own 36-month window.
    #     This is the exact same universe and window as rho -> the tightest possible volatility control,
    #     and it keeps full 1995-2025 coverage because members' pre-1995 history is available.
    _d = (pl.scan_parquet(PANEL)
            .filter(pl.col('is_active') == 1)
            .select(['DlyCalDt', 'DlyRet'])
            .collect(engine='streaming').to_pandas())
    _d['date'] = pd.to_datetime(_d['DlyCalDt'])
    _ew = _d.groupby('date')['DlyRet'].mean().sort_index()         # equal-weight active-universe proxy (daily)
    rv_m = _ew.resample('ME').std() * np.sqrt(252)                 # monthly realized vol, annualized (reactive)

    # formation-universe 36m RV: reuse the exact PIT windows of the signal (Xd from pit_matrices, h_d=1).
    _rv_lh = {}
    for _fm in WINDOWS:                                            # WINDOWS/PIVOT/MEMB/FDATE come from the B3 socle
        _Xd, _Xw, _Xm = pit_matrices(PIVOT, _fm, MEMB, FDATE)      # daily matrix of the 100 members, 36m window
        assert _Xd.shape[1] == 100, f'{_fm:%Y-%m}: expected 100 members, got {_Xd.shape[1]}'
        _ew_win = _Xd.mean(axis=1)                                 # equal-weight daily return of the 100 members
        _key = pd.Timestamp(FDATE[_fm]) + pd.offsets.MonthEnd(0)   # align on the signal's month-end date
        _rv_lh[_key] = _ew_win.std(ddof=1) * np.sqrt(252)
    rv_lh = pd.Series(_rv_lh).sort_index()                         # defined for every rho date

    # assemble; AUDIT coverage on `raw` BEFORE dropping, then build the two blocks separately
    raw = pd.concat({'sig': sig, 'iv': iv_m, 'rv': rv_m, 'iv_lh': iv_lh, 'rv_lh': rv_lh}, axis=1)
    raw.index = pd.to_datetime(raw.index)
    raw = raw[raw.index >= '1995-01-01']                           # restrict to the stated 1995-2025 sample
    coverage = {c: int(raw[c].notna().sum()) for c in raw.columns}
    print('coverage (non-null months) before alignment:', coverage)
    df_m  = raw[['sig', 'iv', 'rv']].dropna()                      # monthly-horizon block
    df_lh = raw[['sig', 'iv_lh', 'rv_lh']].dropna()                # 36-month-horizon block
    assert len(df_m) > 200,  f'monthly block unexpectedly short: {len(df_m)}'
    assert len(df_lh) > 150, f'36-month block unexpectedly short: {len(df_lh)}'
    df = df_m.join(raw[['iv_lh', 'rv_lh']])                        # convenience frame for the overlay
    print(f'monthly block:  {df_m.index.min().date()} -> {df_m.index.max().date()}  (n={len(df_m)})')
    print(f'36-month block: {df_lh.index.min().date()} -> {df_lh.index.max().date()}  (n={len(df_lh)})')

    # --- figure R_13: standardized overlay (signal vs monthly IV/RV) ---
    def _z(s): return (s - s.mean()) / s.std()
    fig, ax = plt.subplots(figsize=(10, 4.2))
    for _, pk in CRISES:
        ax.axvspan(pd.Timestamp(pk)-pd.DateOffset(months=3),
                   pd.Timestamp(pk)+pd.DateOffset(months=3), color=_GREY, alpha=0.10, lw=0)
    ax.plot(df_m.index, _z(df_m['sig']), color=_NAVY,  lw=1.4, label=r'$\sqrt{\widehat{\rho}^{\mathrm{hyb}}_{\tau}}$')
    ax.plot(df_m.index, _z(df_m['iv']),  color=_RUST,  lw=1.0, alpha=0.85, label=r'IV (implied, monthly)')
    ax.plot(df_m.index, _z(df_m['rv']),  color=_GREEN, lw=1.0, alpha=0.75, label=r'RV (realized, monthly)')
    ax.axhline(0, color=_GREY, lw=0.6, ls=':')
    ax.set_xlabel('Date'); ax.set_ylabel('z-score')
    ax.set_title('Dispersion signal vs volatility measures (standardized, 1995–2025)')
    ax.legend(loc='upper left'); ax.margins(x=0.01)
    fig.tight_layout()
    save_fig(fig, 13, 'vol_overlay')
    plt.show()


### 5.1. Persistence and choice of specification

The signal and both volatility measures are slow-moving, highly persistent series. Regressing such series **in levels** risks a *spurious regression* — inflated $R^2$ and $t$-statistics with no stable underlying relationship (Granger–Newbold). Before estimating, we document persistence with the first-order autocorrelation and an augmented Dickey–Fuller (ADF) test on each series.

The signal is a 36-month rolling window and the regressors overlap heavily, so we report inference in both levels and first differences, and use a long Newey–West lag ($36$ months) rather than the short automatic rule. Persistence is documented for **all five series**, including the smoothed 36-month regressors, since those enter the level regressions.

In [ ]:
if RUN_MODE == 'full':
    # 5.b — persistence diagnostics: AR(1) + ADF for ALL FIVE series (short + long horizon)
    SERIES = [(r'$\sqrt{\widehat{\rho}}$', 'sig',   raw['sig']),
              (r'IV (monthly)',            'iv',    raw['iv']),
              (r'RV (monthly)',            'rv',    raw['rv']),
              (r'IV (36m)',                'iv_lh', raw['iv_lh']),
              (r'RV (36m)',                'rv_lh', raw['rv_lh'])]
    _rows = []
    for name, key, s in SERIES:
        s = s.dropna()
        ar1 = s.autocorr(1)
        stat, p, *_ = adfuller(s.values, autolag='AIC')
        _rows.append((name, key, ar1, stat, p))

    # --- clean console recap (no LaTeX) ---
    print('Persistence diagnostics (AR(1) + ADF) on all five series:')
    print(f'  {"series":<8}{"AR(1)":>8}{"ADF":>10}{"p-value":>10}   unit root')
    for name, key, ar1, stat, p in _rows:
        print(f'  {key:<8}{ar1:>8.3f}{stat:>+10.3f}{p:>10.3f}   {"not rejected" if p>=0.05 else "rejected"}')

    # --- export LaTeX table (same filename, English) ---
    body = ''.join(
        f"      {nm} & {ar1:.3f} & ${stat:+.2f}$ & {p:.3f} & {'rejected' if p<0.05 else 'not rejected'} \\\\\n"
        for nm, key, ar1, stat, p in _rows)
    tab = (
        "\\begin{tabular}{lcccc}\n"
        "  \\toprule\n"
        "  Series & AR(1) & ADF stat. & $p$-value & Unit root \\\\\n"
        "  \\midrule\n"
        + body +
        "  \\bottomrule\n"
        "\\end{tabular}\n"
    )
    (TAB/'R_17_vol_persistence.tex').write_text(tab)
    print('\nsaved  tables/R_17_vol_persistence.tex')


### 5.2. Regression at two matched horizons

We estimate, with Newey–West (HAC, $36$-month) standard errors, how much of the signal is spanned by the two volatility measures:
$$\sqrt{\widehat{\rho}_{\tau}} = \alpha + \beta_{\mathrm{IV}}\,\mathrm{IV}_\tau + \beta_{\mathrm{RV}}\,\mathrm{RV}_\tau + u_\tau,$$
reported in **levels** and in **first differences** $\Delta x_\tau = x_\tau - x_{\tau-1}$, at each horizon. The monthly block uses monthly IV/RV; the 36-month block uses the formation-universe realized volatility and the 36-month VIX moving average, matched to the signal's own window. The ADF test does not reject the unit-root null for $\sqrt{\widehat{\rho}}$, while it narrowly rejects it for the 36-month RV; given the extreme persistence and overlapping windows of both series, first differences remain an important robustness specification against spurious level correlation. Since IV and RV are near-collinear, each is also entered **alone**, so that the joint regression can be read without over-interpreting individual coefficients.

In [ ]:
if RUN_MODE == 'full':
    # 5.c — univariate + joint regressions, LEVELS and DIFFERENCES, at two horizons (HAC 36)
    HAC = 36  # Newey-West lag: the signal uses 36-month overlapping windows

    def _fit(dep, regs, frame):
        frame = frame.dropna()
        return sm.OLS(frame[dep], sm.add_constant(frame[regs])).fit(
            cov_type='HAC', cov_kwds={'maxlags': HAC})

    # short (monthly) and long (36m) blocks, in levels and in first differences
    dm, dl   = df_m.copy(), df_lh.copy()
    dm_d, dl_d = dm.diff().dropna(), dl.diff().dropna()

    # R^2 for IV-only, RV-only, IV+RV, in each (horizon x spec) cell
    def _grid(frame, iv_key, rv_key):
        return {'IV':  _fit('sig', [iv_key], frame).rsquared,
                'RV':  _fit('sig', [rv_key], frame).rsquared,
                'both':_fit('sig', [iv_key, rv_key], frame).rsquared}
    R2 = {
        ('monthly',  'levels'): _grid(dm,   'iv',    'rv'),
        ('monthly',  'diffs'):  _grid(dm_d, 'iv',    'rv'),
        ('36-month', 'levels'): _grid(dl,   'iv_lh', 'rv_lh'),
        ('36-month', 'diffs'):  _grid(dl_d, 'iv_lh', 'rv_lh'),
    }
    # joint coefficients (levels) to expose the collinearity sign flip
    jm, jl = _fit('sig', ['iv', 'rv'], dm), _fit('sig', ['iv_lh', 'rv_lh'], dl)
    corr_m, corr_l = dm['iv'].corr(dm['rv']), dl['iv_lh'].corr(dl['rv_lh'])

    # --- clean console recap (no LaTeX) ---
    def _st(p): return '***' if p < .01 else '**' if p < .05 else '*' if p < .1 else ''
    W = 62
    print('=' * W)
    print('  R²  —  fraction of √ρ explained by volatility   (HAC lags = %d)' % HAC)
    print('=' * W)
    print(f'  corr(IV, RV):  monthly = {corr_m:.3f}   36-month = {corr_l:.3f}   (near-collinear)')
    print('-' * W)
    print(f'  {"horizon":<10}{"spec":<12}{"R²(IV)":>8}{"R²(RV)":>8}{"R²(IV+RV)":>11}')
    print('-' * W)
    for (hz, spec), g in R2.items():
        tail = '   ← matched horizon' if (hz == '36-month' and spec == 'levels') else ''
        print(f'  {hz:<10}{spec:<12}{g["IV"]:>8.3f}{g["RV"]:>8.3f}{g["both"]:>11.3f}{tail}')
    print('-' * W)
    _r2_key = R2[('36-month', 'levels')]['both']
    print(f'  KEY RESULT (matched 36-month horizon, levels):')
    print(f'    volatility explains {_r2_key*100:.1f}% of the signal;'
          f' residual = {(1-_r2_key)*100:.1f}%')
    print('-' * W)
    print('  joint coefficients (levels):')
    print(f'    monthly    IV = {jm.params["iv"]:+.5f} (t={jm.tvalues["iv"]:+.2f}{_st(jm.pvalues["iv"])})'
          f'   RV = {jm.params["rv"]:+.5f} (t={jm.tvalues["rv"]:+.2f}{_st(jm.pvalues["rv"])})')
    print(f'    36-month   IV = {jl.params["iv_lh"]:+.5f} (t={jl.tvalues["iv_lh"]:+.2f}{_st(jl.pvalues["iv_lh"])})'
          f'   RV = {jl.params["rv_lh"]:+.5f} (t={jl.tvalues["rv_lh"]:+.2f}{_st(jl.pvalues["rv_lh"])})')
    _gain_rv = R2[('36-month','levels')]['both'] - R2[('36-month','levels')]['IV']   # RV on top of IV
    _gain_iv = R2[('36-month','levels')]['both'] - R2[('36-month','levels')]['RV']   # IV on top of RV
    print('  reading: IV and RV each explain the signal ALONE (near-collinear).')
    print(f'           At 36m/levels the joint gain is asymmetric: +{_gain_rv:.3f} adding RV on IV,')
    print(f'           +{_gain_iv:.3f} adding IV on RV — a shared volatility regime, not two')
    print('           independent contributions.')
    print('=' * W)

    # --- export LaTeX table: clean, readable, booktabs, both horizons x (levels, diffs) ---
    def _r(hz, spec, k): return f"{R2[(hz, spec)][k]:.3f}"
    def _cc(m, key):
        st = '^{***}' if m.pvalues[key] < .01 else '^{**}' if m.pvalues[key] < .05 else '^{*}' if m.pvalues[key] < .1 else ''
        return f"${m.params[key]:+.4f}{st}$"
    tab = (
        "\\begin{tabular}{@{}l cc cc@{}}\n"
        "  \\toprule\n"
        "  & \\multicolumn{2}{c}{Monthly horizon} & \\multicolumn{2}{c}{36-month horizon} \\\\\n"
        "  \\cmidrule(lr){2-3}\\cmidrule(lr){4-5}\n"
        "  & Levels & Differences & Levels & Differences \\\\\n"
        "  \\midrule\n"
        "  \\multicolumn{5}{@{}l}{\\emph{$R^2$: signal explained by\\ldots}} \\\\\n"
        f"  \\quad implied vol.\\ (IV) alone   & {_r('monthly','levels','IV')}  & {_r('monthly','diffs','IV')}  & {_r('36-month','levels','IV')}  & {_r('36-month','diffs','IV')}  \\\\\n"
        f"  \\quad realized vol.\\ (RV) alone  & {_r('monthly','levels','RV')}  & {_r('monthly','diffs','RV')}  & {_r('36-month','levels','RV')}  & {_r('36-month','diffs','RV')}  \\\\\n"
        f"  \\quad IV and RV jointly          & {_r('monthly','levels','both')}& {_r('monthly','diffs','both')}& {_r('36-month','levels','both')}& {_r('36-month','diffs','both')}\\\\\n"
        "  \\addlinespace\n"
        "  \\multicolumn{5}{@{}l}{\\emph{Joint coefficients (levels, HAC)}} \\\\\n"
        f"  \\quad IV                         & \\multicolumn{{2}}{{c}}{{{_cc(jm,'iv')}}} & \\multicolumn{{2}}{{c}}{{{_cc(jl,'iv_lh')}}} \\\\\n"
        f"  \\quad RV                         & \\multicolumn{{2}}{{c}}{{{_cc(jm,'rv')}}} & \\multicolumn{{2}}{{c}}{{{_cc(jl,'rv_lh')}}} \\\\\n"
        "  \\midrule\n"
        f"  corr(IV, RV)                     & \\multicolumn{{2}}{{c}}{{{corr_m:.3f}}} & \\multicolumn{{2}}{{c}}{{{corr_l:.3f}}} \\\\\n"
        f"  $n$                              & {int(jm.nobs)} & {len(dm_d)} & {int(jl.nobs)} & {len(dl_d)} \\\\\n"
        "  \\bottomrule\n"
        "\\end{tabular}\n"
    )
    (TAB/'R_18_vol_regression.tex').write_text(tab)
    print('\nsaved  tables/R_18_vol_regression.tex   (HAC lag =', HAC, ')')


### 5.3. Interpretation

The picture is consistent once the comparison is put on the same time scale, and it must be read across **both** the level and the difference specifications. The ADF test does not reject the unit-root null for $\sqrt{\widehat{\rho}}$ and only narrowly rejects it for the 36-month RV; given the extreme persistence of both series, differencing is an essential control against spurious level correlation, not a secondary output.

- **In levels**, at the matched 36-month horizon the volatility proxies explain about **three quarters** of the persistent variation in the signal ($R^2 \approx 0.75$).
- **In first differences**, which mitigate the spurious-correlation concern, they explain **approximately 62%** (about three fifths) of its month-to-month variation — and there almost entirely through **realized volatility**, the implied-volatility average adding little once RV is included.

Under neither specification is the signal **fully linearly spanned** by the proxies. Two nuances qualify the joint reading. First, implied and realized volatility are **near-collinear** (correlation $\approx 0.9$), so the individual coefficients — including the sign flip on RV in the *monthly-level* regression — reflect collinearity rather than an inverse economic relation. Second, the joint gain is *asymmetric*: at the 36-month level horizon, adding RV on top of IV raises $R^2$ only marginally, whereas adding IV on top of RV is a more modest but non-trivial gain. The common component is best described as a **shared, persistent volatility regime** captured by either proxy, not a component causally attributable to one of them.

The residual — roughly a quarter of the level variation, and **approximately 38%** of the differenced variation — is **compatible with**, though not by itself proof of, the signal capturing cross-frequency distributional disagreement beyond a conventional scale effect. Whether this residual component carries incremental predictive or economic value is left for future research.